# ~~~~ VISUALISATION WFLOW OUTPUT ~~~~

In [6]:
using CSV, Dates, Tables, CairoMakie, GLMakie,  NCDatasets, Dates


Path_Home = @__DIR__
cd(Path_Home)

include("Julia//Parameters.jl")


:LEVEL_2_ID

##	PARAMETERS

In [43]:
ΔX = 5 # m
ΔY = 5 # m

StartYear    = 2021
StartMonth    = 01
StartDay      = 01
StartDate = Date(StartYear, StartMonth, StartDay)

EndYear    = 2021
EndMonth    = 07
EndDay      = 01
EndDate = Date(EndYear, EndMonth, EndDay)



2021-07-01

## Read input flowcdata

# Averaging output.netcdf_grid

In [61]:
Path_Wflow_NetCDF_Output = joinpath(Path_Root_NetCDF_Output, "output_" * 🎏_CatchmentName * ".nc")

@assert isfile(joinpath(Path_Wflow_NetCDF_Output))

WflowOutput = NCDatasets.NCDataset(Path_Wflow_NetCDF_Output)

Precipitation        = Array(WflowOutput["Precipitation"])
PotentialEvaporation = Array(WflowOutput["PotentialEvaporation"])
Interception         = Array(WflowOutput["Interception"])
close(WflowOutput)

Nx, Ny, Nt = size(Precipitation)

Precipitation_Average        = fill(0.0, Nt)
PotentialEvaporation_Average = fill(0.0, Nt)
Interception_Average         = fill(0.0, Nt)
ReachSoil_Average            = fill(0.0, Nt)
NotMissing                   = fill(false, Nx, Ny)
DatesWflow                   = fill(Date(StartYear, StartMonth, StartDay), Nt)

# Counting active cells
	CountCell = 0
	 for iX=1:Nx, iY=1:Ny
		if Precipitation[iX, iY, 2] !== missing
			CountCell += 1
			NotMissing[iX, iY] = true
		end
	end

	CatchmentArea = CountCell * Param_ΔX₁ * Param_ΔX₁  # m

	printstyled("	Number of grids = $CountCell , CatchmentSize = $CatchmentArea  m² \n", color = :green)

	DatesWflow[1] = StartDate

for iT=1:Nt
	 for iX=1:Nx
		for iY=1:Ny
			if NotMissing[iX, iY]
				Precipitation_Average[iT] += Precipitation[iX, iY, iT]
				PotentialEvaporation_Average[iT] += PotentialEvaporation[iX, iY, iT]
				Interception_Average[iT] += Interception[iX, iY, iT]
			end
		end
	end # for iX=1:Nx, iY=1:Ny

   Precipitation_Average[iT]        = Precipitation_Average[iT] / CountCell
   PotentialEvaporation_Average[iT] = PotentialEvaporation_Average[iT] / CountCell
   Interception_Average[iT]         = Interception_Average[iT] / CountCell

	ReachSoil_Average[iT] = Precipitation_Average[iT] - Interception_Average[iT]

	DatesWflow[iT] +=  Dates.Day(iT)
end # for iT=1:Nt

InterceptionLost = 100.0 * sum(Interception_Average) / sum(Precipitation_Average)
printstyled("	Interception lost = $(floor(InterceptionLost)) %  \n", color = :green)


	Number of grids = 324770 , CatchmentSize = 8119250  m² 
	Interception lost = 11.0 %  


# Reading Qriver from Wflow output


# Reading Q observed

In [66]:
Path_TimeSeries = joinpath(Path_Root, Path_Forcing, Filename_Input_Forcing)
println(Path_TimeSeries)
@assert isfile(Path_TimeSeries)

# Reading climate data
	Data₀  = CSV.File(Path_TimeSeries; header=true)
	Year   = convert(Vector{Int64}, Tables.getcolumn(Data₀, :Year))
	Month  = convert(Vector{Int64}, Tables.getcolumn(Data₀, :Month))
	Day    = convert(Vector{Int64}, Tables.getcolumn(Data₀, :Day))
	Hour   = convert(Vector{Int64}, Tables.getcolumn(Data₀, :Hour))

	Qobs₀  = convert(Vector, Tables.getcolumn(Data₀, :RiverDischarge_cumec))

	Time_Forcing = Date.(Year, Month, Day) #  <"standard"> "proleptic_gregorian" calendar

# Selecting time which is between Start_DateTime and End_DateTime
	True = fill(false::Bool, length(Year))
	for iT=1:length(Year)
		if (DatesWflow[1] ≤ Time_Forcing[iT] ≤ DatesWflow[end])
			True[iT] = true
		end
		if Time_Forcing[iT] > EndDate
			break
		end
	end # for iT=1:Nit

	Qobs = Qobs₀[True[:]];

	Qobs_mm = (Qobs ./ CatchmentArea) * 1000.0; # cumec to mm/ m2 / day


D:/JOE/MAIN/MODELS/WFLOW/DATA/CATCHMENTS/Timoleague\InputTimeSeries\TimeSeries_Process\Daily\Forcing_Daily_Timoleague.csv


In [67]:
# Water balance
# [Precipitation_Average] - Interception_Average[iT] = Stemflow_Average[iT] + Throughfall_Average[iT]

Path_AverageFluxes_Csv₀ = joinpath(Path_Root, Path_WflowModelOutput)
mkpath(Path_AverageFluxes_Csv₀)

Path_AverageFluxes_Csv = joinpath(Path_AverageFluxes_Csv₀, Filename_WflowCatchementAverage_Csv)

Header = ["Date", "Precipitation[mm]", "PotentialEvaporation[mm]", "Interception[mm]", "ThroughfallStemflow[mm]",  "Qobs[cumec]", "Qobs_mm[cumec]"]
CSV.write(Path_AverageFluxes_Csv, Tables.table([DatesWflow Precipitation_Average PotentialEvaporation_Average Interception_Average ReachSoil_Average Qobs Qobs_mm]), writeheader=true, header=Header, bom=true)




"D:/JOE/MAIN/MODELS/WFLOW/DATA/CATCHMENTS/Timoleague\\WflowModelOutput\\CatchementAverage_Timoleague.csv"